In [ ]:
!pip install langdetect

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 9.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for langdetect: filename=langdetect-1.0.9-py3-none-any.whl size=993223 sha256=22e662665b764caa4ae576498c24b67e7478ace99f3f4d6acbd7ad5aa43a987d
  Stored in directory: /root/.cache/pip/wheels/c1/67/88/e844b5b022812e15a52e4eaa38a1e709e99f06f6639d7e3ba7
Successfully built langdetect


In [ ]:
import csv
import re
import pandas as pd
import nltk
import numpy as np
import joblib
from langdetect import detect, detect_langs, DetectorFactory
from langdetect.lang_detect_exception import LangDetectException
from tqdm.auto import tqdm

from nltk.stem import WordNetLemmatizer
from nltk.corpus import wordnet
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
from sklearn.feature_extraction import text
from sklearn.metrics import confusion_matrix

tqdm.pandas()
nltk.download('averaged_perceptron_tagger_eng')
nltk.download('wordnet')
nltk.download('omw-1.4')

[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger_eng.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...


True

In [ ]:
file_path = '/content/drive/MyDrive/Colab Notebooks/NLP_Ctrl-Alt-Elite/data/dataset.csv'
df = pd.read_csv(file_path)

In [ ]:
df.head()

,label,text
0,1,Congratulations! You've been selected for a lu...
1,1,URGENT: Your account has been compromised. Cli...
2,1,You've won a free iPhone! Claim your prize by ...
3,1,Act now and receive a 50% discount on all purc...
4,1,Important notice: Your subscription will expir...


In [ ]:
#Language detection
DetectorFactory.seed = 0

def lang_detect(text):
  if not isinstance(text,str) or not text.strip():
    return 'Unknown'
  try:
    return detect(text)
  except LangDetectException:
    return 'Unknown'

In [ ]:
#detect language
df['language'] = df['text'].progress_apply(lang_detect)
#filter for english
df = df[df['language'] == 'en'].copy()

  0%|          | 0/28232 [00:00<?, ?it/s]

In [ ]:
#regex Cleaning
def regex_clean(text):
    if not isinstance(text, str): # Ensure text is a string
        return ''

    # 1. Lowercase the text
    text = text.lower()

    # 2. Remove HTML tags
    text = re.sub(r'<[^>]+>', ' ', text)

    # 3. Remove URLs/Hyperlinks
    text = re.sub(r'https?://\S+|www\.\S+', ' ', text)

    # 4. Remove Email Addresses
    text = re.sub(r'\S+@\S+', ' ', text)

    # 5. Remove Numbers (often randomized in spam)
    text = re.sub(r'\d+', ' ', text)

    # 6. Remove Punctuation and Special Characters
    text = re.sub(r'[^\w\s]', ' ', text)

    # 7. Collapse multiple spaces into a single space
    text = re.sub(r'\s+', ' ', text).strip()

    return text

# Apply the cleaning function to the 'text' column to create 'cleaned_text'
df['cleaned_text'] = df['text'].progress_apply(regex_clean)

  0%|          | 0/26487 [00:00<?, ?it/s]

In [ ]:
#Additional cleaning - drop duplicates
df.drop_duplicates(subset=['cleaned_text'], keep='first')
df.rename(columns={'text': 'raw_text'},inplace=True)

In [ ]:
lemmatizer = WordNetLemmatizer()
#POS tagging
def get_wordnet_pos_from_tag(tag):
    if not tag:
        return wordnet.NOUN
    first_letter = tag[0].upper()
    tag_dict = {
        "J": wordnet.ADJ,
        "N": wordnet.NOUN,
        "V": wordnet.VERB,
        "R": wordnet.ADV
    }
    return tag_dict.get(first_letter, wordnet.NOUN)

#Lemmatization
def lemmatize_text(text):
    if not isinstance(text, str):
        return ""
    words = text.split()
    word_tags = nltk.pos_tag(words) # Tags the whole sentence once

    lemmatized_words = [
        lemmatizer.lemmatize(word, get_wordnet_pos_from_tag(tag))
        for word, tag in word_tags
    ]
    return " ".join(lemmatized_words)

In [ ]:
df['cleaned_lemmatized'] = df['cleaned_text'].progress_apply(lemmatize_text)

  0%|          | 0/26487 [00:00<?, ?it/s]

In [ ]:
#exporting cleaned dataset for ease of use
df.to_csv('/content/drive/MyDrive/Colab Notebooks/NLP_Ctrl-Alt-Elite/data/cleaned_dataset.csv', columns=['cleaned_lemmatized', 'label'] , index=False)

In [ ]:
#Model training - Logistic regression
X_train, X_test, y_train, y_test = train_test_split(
    df['cleaned_lemmatized'],
    df['label'],
    test_size=0.2,
    random_state=42
)

#Stop word removal + step to add custom stop words detected while developping nlp pipeline
custom_stop_words = list(text.ENGLISH_STOP_WORDS) #+ ['http', 'com', 'subject', 'just', 'email', 'www'] #Additional stop words removal - detected unremovable by cleaning steps
vectorizer = TfidfVectorizer(stop_words=custom_stop_words, max_features=5000)

#Vectorization
X_train_vectorized = vectorizer.fit_transform(X_train)
X_test_vectorized = vectorizer.transform(X_test)

#Model fit
model = LogisticRegression(max_iter=1000)
model.fit(X_train_vectorized, y_train)

predictions = model.predict(X_test_vectorized)

print("--- Logistic Regression Accuracy ---")
print(f"{accuracy_score(y_test, predictions) * 100:.2f}%")

print("\n--- Detailed Performance Report ---")
print(classification_report(y_test, predictions))

# 5. Extract top spam words using Logistic Regression weights
words = vectorizer.get_feature_names_out()
# Logistic Regression stores word importance weights in .coef_[0]
spam_word_weights = model.coef_[0]
top_10_indices = np.argsort(spam_word_weights)[-10:]
top_10_words = [words[i] for i in top_10_indices]

print("\n--- Logistic Regression Top 10 Spam Words (Standard Stop Words) ---")
print(top_10_words[::-1])


--- Logistic Regression Accuracy ---
96.49%

--- Detailed Performance Report ---
              precision    recall  f1-score   support

           0       0.96      0.99      0.98      4209
           1       0.97      0.86      0.91      1089

    accuracy                           0.96      5298
   macro avg       0.97      0.93      0.94      5298
weighted avg       0.96      0.96      0.96      5298


--- Logistic Regression Top 10 Spam Words (Standard Stop Words) ---
['http', 'claim', 'mobile', 'subject', 'txt', 'click', 'tone', 'sex', 'remove', 'software']


In [ ]:
#export model and vectorizer
joblib.dump(model, '/content/drive/MyDrive/Colab Notebooks/NLP_Ctrl-Alt-Elite/Models/logistic_regression_model.pkl')
joblib.dump(vectorizer, '/content/drive/MyDrive/Colab Notebooks/NLP_Ctrl-Alt-Elite/Models/tfidf_vectorizer.pkl')

['/content/drive/MyDrive/Colab Notebooks/NLP_Ctrl-Alt-Elite/Models/tfidf_vectorizer.pkl']

In [ ]:
#import validation dataset
df_validation = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/NLP_Ctrl-Alt-Elite/data/validation_dataset.csv')

In [ ]:
#applying same nlp pipeline for validation dataset
df_validation.rename(columns={'Email Text': 'raw_text', 'Email Type': 'label'},inplace=True)
df_validation['label'] = df_validation['label'].map({'Phishing Email': 1, 'Safe Email': 0})
df_validation['cleaned_text'] = df_validation['raw_text'].progress_apply(regex_clean)
df_validation['cleaned_lemmatized'] = df_validation['cleaned_text'].progress_apply(lemmatize_text)

  0%|          | 0/2000 [00:00<?, ?it/s]

  0%|          | 0/2000 [00:00<?, ?it/s]

In [ ]:
#evaluating accuracy
validation_text = df_validation['cleaned_lemmatized']
y_val = df_validation['label']

X_val = vectorizer.transform(validation_text)

predictions = model.predict(X_val)
accuracy = accuracy_score(y_val, predictions)

print(f"Validation Accuracy: {accuracy * 100:.2f}%")
print("\nClassification Report")
print(classification_report(y_val, predictions))
print("\nConfusion Matrix")
print(confusion_matrix(y_val, predictions))

Validation Accuracy: 80.05%

Classification Report
              precision    recall  f1-score   support

           0       0.71      1.00      0.83      1000
           1       1.00      0.60      0.75      1000

    accuracy                           0.80      2000
   macro avg       0.86      0.80      0.79      2000
weighted avg       0.86      0.80      0.79      2000


Confusion Matrix
[[1000    0]
 [ 399  601]]


In [ ]:
#checking cause of accuracy drop
print(df['label'].value_counts(normalize=True))
print(df_validation['label'].value_counts(normalize=True))

label
0    0.791181
1    0.208819
Name: proportion, dtype: float64
label
0    0.5
1    0.5
Name: proportion, dtype: float64
